# D1 pipeline datasheet: what the pinned snapshots can support

This executable datasheet audits the **offline, DVC-pinned inputs** consumed by VINE's model tracks. It asks four practical questions before any modeling:

1. Which sensor channels exist, over what dates, and at what cadence?
2. Where are the hourly gaps, and are missing periods shared across devices?
3. Does daily weather—including reference evapotranspiration (ET₀)—cover the sensor timeline after the production join?
4. Which of the 39 vineyard blocks have a deployed probe and metadata indicating usable imagery?

The notebook is narrative-first and intentionally reads no live services. Its figures describe availability, not biological validity or model performance.

## Reproduce this audit offline

From a checkout whose DVC cache has already been populated, restore the exact inputs with `uv run dvc pull`, then use **Restart & Run All**. Every data read below is from `data/raw/`; no InfluxDB, Open-Meteo, STAC, or NextCloud client is constructed.

The committed notebook is output-free. CI executes a disposable copy and discards its outputs:

```bash
uv run python scripts/check_notebooks.py notebooks/02_pipeline_datasheet.ipynb
```

Missing snapshots fail loudly rather than triggering a network fallback.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from vine.d1_pipeline.datasheet import (
    block_alignment_summary,
    dvc_snapshot_manifest,
    read_imagery_inventory,
    select_deployed_points,
    sensor_coverage,
    weather_coverage,
    weekly_missingness,
)
from vine.d1_pipeline.geo import load_blocks_kmz, load_points_kmz
from vine.d1_pipeline.ingest import load_snapshot, load_weather_snapshot
from vine.d1_pipeline.pipeline import attach_weather, build_sensor_features

ROOT = Path.cwd()
RAW = ROOT / "data" / "raw"
SENSORS = RAW / "sensors"
IMAGERY = RAW / "imagery"

required = [
    RAW / "sensors.dvc",
    RAW / "weather.dvc",
    RAW / "imagery.dvc",
    SENSORS,
    RAW / "weather",
    IMAGERY / "inventory.parquet",
    IMAGERY / "IHV-2026-05-26.kmz",
]
missing = [str(path.relative_to(ROOT)) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Pinned inputs are missing; run `uv run dvc pull`: {missing}")

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
        "axes.labelcolor": "#303030",
        "text.color": "#303030",
    }
)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 30)

## 1. Provenance is part of the dataset

A snapshot is identified by its DVC directory hash—not merely by a filename or download date. The manifest below is read directly from the three tracked pointer files. Sizes are compressed/storage metadata from DVC and are not row counts.

This audit does **not** establish custody before ingestion, sensor calibration, or field interpretation. It establishes which immutable local snapshot was analyzed.

In [ ]:
dvc_paths = [RAW / "sensors.dvc", RAW / "weather.dvc", RAW / "imagery.dvc"]
provenance = dvc_snapshot_manifest(dvc_paths)
provenance["pointer"] = provenance["pointer"].map(lambda value: str(Path(value).relative_to(ROOT)))
provenance["size_mib"] = provenance["size_bytes"] / 1024**2
display(
    provenance[["pointer", "path", "md5", "nfiles", "size_mib"]]
    .sort_values("path")
    .style.format({"size_mib": "{:.2f}"})
)

### Known unknowns: do not turn availability into semantics

- **EM500-PP-4842:** the pinned column is `pipe_pressure_raw`. Its engineering unit, active direction, served block, and relationship to a real irrigation event are unverified. It is shown for completeness but must not be used as an irrigation-event label.
- **SE0X-LS-1:** its channels retain raw LoRa field names (including the `SOIL1` suffix). Probe depth, calibration, and engineering units still require field confirmation; this notebook does not silently rename or normalize them.
- **Gaps:** a missing hourly bin means no observation reached this snapshot. It does not distinguish probe failure, gateway outage, maintenance, or a true environmental state.
- **Weather:** Open-Meteo archive values are gridded historical estimates at vineyard coordinates, not an on-site weather-station measurement. ET₀ coverage is not ET₀ validation.
- **Imagery:** `available=True` is metadata inventory evidence. It does not prove cloud-free pixels, valid vegetation, geometric alignment quality, or a stress/pest label.
- Historical irrigation, harvest, yield, Brix, pH, TA, and supervised plant-health labels are absent from these pinned inputs. Their existence and semantics must be confirmed with the mentor.

## 2. Sensor coverage and cadence

Snapshots are loaded through `vine.d1_pipeline.ingest.load_snapshot`. Coverage is then computed by `vine.d1_pipeline.datasheet.sensor_coverage`, which regularizes each numeric channel to an hourly grid without imputing values. Raw cadence summarizes arrival spacing; hourly completeness summarizes analysis-ready bins. Those are different quantities and should not be conflated.

In [ ]:
sensor_files = sorted(SENSORS.glob("*.parquet"))
frames = {path.stem: load_snapshot(path.stem, SENSORS) for path in sensor_files}
if not frames:
    raise FileNotFoundError("No pinned sensor Parquet snapshots found")

coverage = sensor_coverage(frames, freq="1h")
coverage_table = coverage[
    [
        "device",
        "channel",
        "raw_rows",
        "start",
        "end",
        "median_cadence_min",
        "p90_cadence_min",
        "observed_bins",
        "expected_bins",
        "missing_pct",
    ]
].sort_values(["device", "channel"])

display(
    coverage_table.style.format(
        {
            "median_cadence_min": "{:.1f}",
            "p90_cadence_min": "{:.1f}",
            "missing_pct": "{:.1f}%",
        }
    )
)

### Gap table

The longest-gap count is in consecutive **hourly bins**. Sorting by missing percentage surfaces the weakest channel/device combinations while preserving every channel in the table. Shared missing periods are visible more clearly in the weekly heatmap that follows.

In [ ]:
gap_table = coverage[
    ["device", "channel", "missing_bins", "missing_pct", "longest_gap_bins"]
].sort_values(["missing_pct", "longest_gap_bins"], ascending=False)
display(gap_table.style.format({"missing_pct": "{:.1f}%"}))

### Weekly missingness: one representative channel per device

To keep the heatmap interpretable, each row uses one declared representative channel: temperature for air probes, soil water for soil probes, and raw pressure for EM500-PP-4842. A dark cell means a larger fraction of expected hourly bins was missing. Gray means the week falls outside that device's represented timeline, not 100% missingness.

In [ ]:
representative_channels = {
    device: (
        "pipe_pressure_raw"
        if "pipe_pressure_raw" in frame.columns
        else "soil_water"
        if "soil_water" in frame.columns
        else "device_frmpayload_data_water_SOIL1"
        if "device_frmpayload_data_water_SOIL1" in frame.columns
        else "temperature"
    )
    for device, frame in frames.items()
}
weekly = weekly_missingness(frames, channels=representative_channels, freq="1h")
heat = weekly.pivot(index="device", columns="week", values="missing_fraction").sort_index()

cmap = plt.get_cmap("cividis").copy()
cmap.set_bad("#e6e6e6")
fig, ax = plt.subplots(figsize=(13, 5.5), constrained_layout=True)
image = ax.imshow(heat.to_numpy(), aspect="auto", vmin=0, vmax=1, cmap=cmap)
ax.set_title("Weekly missingness on the hourly grid")
ax.set_xlabel("Week ending Monday (UTC)")
ax.set_ylabel("Device")
ax.set_yticks(np.arange(len(heat.index)), heat.index)
step = max(1, len(heat.columns) // 12)
ticks = np.arange(0, len(heat.columns), step)
ax.set_xticks(ticks, [heat.columns[i].strftime("%b %d") for i in ticks], rotation=45, ha="right")
colorbar = fig.colorbar(image, ax=ax, fraction=0.025, pad=0.02)
colorbar.set_label("Missing fraction")
plt.show()

## 3. Historical weather and ET₀ join coverage

The weather snapshot is loaded with `load_weather_snapshot`. First, `weather_coverage` asks whether daily precipitation and ET₀ exist over the union of sensor dates. Then the notebook runs the actual D1 path—`build_sensor_features` followed by `attach_weather`—for each representative channel and reports non-null coverage on that device's hourly grid.

Forward-filling a daily value onto hourly rows is a join convention, not imputation of missing sensor readings. Hours beyond the last pinned weather day remain uncovered.

In [ ]:
weather = load_weather_snapshot(RAW)
if weather is None:
    raise FileNotFoundError("No pinned historical weather snapshot found")

all_sensor_times = pd.DatetimeIndex(
    np.concatenate([frame.index.to_numpy() for frame in frames.values()])
).sort_values()
daily_join_coverage = weather_coverage(
    all_sensor_times,
    weather,
    columns=("precip_mm", "et0_mm"),
)
display(daily_join_coverage.style.format({"coverage_pct": "{:.1f}%"}))

join_rows = []
for device, raw in sorted(frames.items()):
    channel = representative_channels[device]
    hourly = build_sensor_features(
        raw,
        value_cols=[channel],
        freq="1h",
        rolling_windows=(),
        lags=(),
    )
    joined = attach_weather(hourly, weather)
    join_rows.append(
        {
            "device": device,
            "sensor_start": hourly.index.min(),
            "sensor_end": hourly.index.max(),
            "hourly_rows": len(hourly),
            "precip_join_pct": 100 * joined["precip_mm"].notna().mean(),
            "et0_join_pct": 100 * joined["et0_mm"].notna().mean(),
        }
    )
join_table = pd.DataFrame(join_rows)
display(join_table.style.format({"precip_join_pct": "{:.1f}%", "et0_join_pct": "{:.1f}%"}))

## 4. Vineyard blocks, deployed probes, and imagery inventory

The KMZ is the spatial authority for this audit. VINE's geo loaders parse all polygon and point placemarks, and `block_alignment_summary` performs the same point-in-polygon assignment used by the D1 pipeline. An unmatched probe remains explicitly unassigned.

Imagery is **metadata-only** here: the inventory Parquet is summarized without opening or downloading a raster. The 2025-08-29 acquisition covers a subset of H-area blocks, while the 2026-06-01 inventory lists whole-vineyard orthomosaics. “Available blocks” counts metadata rows marked available; it is not pixel coverage.

In [ ]:
kmz = IMAGERY / "IHV-2026-05-26.kmz"
blocks = load_blocks_kmz(kmz)
points = load_points_kmz(kmz)
if len(blocks) != 39:
    raise AssertionError(f"Expected 39 vineyard blocks, found {len(blocks)}")

inventory = read_imagery_inventory(IMAGERY / "inventory.parquet")
inventory_summary = (
    inventory.assign(available_size_bytes=inventory["size_bytes"].where(inventory["available"], 0))
    .groupby(["acquisition", "asset_kind", "band"], dropna=False)
    .agg(
        listed_blocks=("block_id", "nunique"),
        available_blocks=("available", "sum"),
        available_size_bytes=("available_size_bytes", "sum"),
    )
    .reset_index()
)
inventory_summary["available_size_mib"] = inventory_summary["available_size_bytes"] / 1024**2
display(
    inventory_summary[
        [
            "acquisition",
            "asset_kind",
            "band",
            "listed_blocks",
            "available_blocks",
            "available_size_mib",
        ]
    ].style.format({"available_size_mib": "{:.1f}"})
)

### Spatial overlay

Block fill encodes the number of acquisition dates for which the inventory marks **both NDVI and NDRE** available. Orange circles are point placemarks whose names exactly match pinned sensor device IDs. The companion table is authoritative for probe-to-block assignment; repeated placemarks remain visible as repeated rows rather than being silently deduplicated.

In [ ]:
device_ids = sorted(frames)
alignment = block_alignment_summary(blocks, points, device_ids)
deployed = select_deployed_points(points, device_ids)

index_inventory = inventory[inventory["available"] & inventory["band"].isin(["NDVI", "NDRE"])]
complete_index_dates = index_inventory.groupby(["block_id", "acquisition"])["band"].nunique().eq(2)
imagery_dates = (
    complete_index_dates[complete_index_dates]
    .groupby(level="block_id")
    .size()
    .rename("imagery_acquisitions")
)
map_blocks = blocks.merge(imagery_dates, left_on="block_id", right_index=True, how="left")
map_blocks["imagery_acquisitions"] = map_blocks["imagery_acquisitions"].fillna(0).astype(int)

palette = ListedColormap(["#f2f2f2", "#9ecae1", "#2171b5"])
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], palette.N)
fig, ax = plt.subplots(figsize=(10, 10), constrained_layout=True)
map_blocks.plot(
    ax=ax,
    column="imagery_acquisitions",
    cmap=palette,
    norm=norm,
    edgecolor="#4d4d4d",
    linewidth=0.8,
)
deployed.plot(
    ax=ax,
    marker="o",
    color="#eb6834",
    edgecolor="white",
    linewidth=0.7,
    markersize=42,
    zorder=3,
)
for row in map_blocks.itertuples():
    point = row.geometry.representative_point()
    ax.annotate(
        row.block_id,
        (point.x, point.y),
        ha="center",
        va="center",
        fontsize=6.5,
        color="#202020",
    )
ax.set_title("All 39 vineyard blocks: deployed probes and metadata imagery coverage")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(
    handles=[
        Patch(facecolor="#f2f2f2", edgecolor="#4d4d4d", label="0 complete index dates"),
        Patch(facecolor="#9ecae1", edgecolor="#4d4d4d", label="1 complete index date"),
        Patch(facecolor="#2171b5", edgecolor="#4d4d4d", label="2 complete index dates"),
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor="#eb6834",
            markeredgecolor="white",
            markersize=7,
            label="Pinned deployed probe",
        ),
    ],
    loc="upper left",
    frameon=True,
    title="Overlay",
)
plt.show()

display(alignment.sort_values(["device", "block_id"], na_position="last"))

## What this snapshot supports—and what it does not

The pinned data support reproducible sensor-quality profiling, historical weather/ET₀ joins, block-level probe alignment, and metadata-level imagery availability checks. They do **not** by themselves support causal attribution of gaps, pressure-derived irrigation events, calibrated cross-probe comparisons, supervised plant-health classification, or harvest/yield modeling.

Before those claims are made, field owners must confirm sensor units and placement (especially EM500-PP-4842 and SE0X-LS-1), provide event/harvest/health labels, and validate imagery at pixel level. Until then, VINE should preserve raw names, explicit gaps, unmatched points, and DVC hashes exactly as shown here.